# **Surprise Labeling Pipeline**
A rule-based labeling system that detects and measures surprise intensity in weighted text comments.

## Overview
Third-stage pipeline that analyzes category-weighted comments and assigns surprise labels with intensity levels (0-3). Uses multi-category support mapping and configurable thresholds for precise semantic detection.

## Key Features

*   **Binary surprise detection**: Identifies comments containing surprise via category combinations
*   **Intensity scaling**: Three-level labeling (weak/medium/strong) based on weight thresholds
*   **Support map logic**: Detects implicit surprise through related categories (Strangeness→Surprise, Interest→Unexpectedness, etc.)
*   **Category restrictions**: Caps intensity for limited-category comments (e.g., only "Strangeness" max level 1)


## Core Components

*   **SurpriseConfig**: Centralised configuration with thresholds, restricted categories, and support maps
*   **determine_surprise()**: Binary classifier using support map and threshold checks
*   **determine_surprise_intensity()**: Three-level intensity classifier with special-case handling
*   **Main pipeline**: End-to-end processing with progress bars and distribution logging

## Input Requirements
*   **Input CSV**: Must contain weight_{category} columns (output from Weight Calculation Pipeline)
*   **Text column**: text field for cleaning and preservation

## Output
Generates enriched DataFrame with:

*   is_surprise: Binary flag (0/1)
*   surprise_intensity: Intensity level (0=no surprise, 1=weak, 2=medium, 3=strong)

## Intensity Logic
*  **Level 3**: Direct "Amazement" OR Surprise≥3.0 OR Total≥6.0 (unless only restricted categories)
*  **Level 2**: Support map match OR Surprise≥2.0 OR Total≥3.0
*  **Level 1**: All other detected surprises (capped if only "Strangeness/Deviations" & "Interest")

## Technical Details
Built with pandas, NumPy, and tqdm. Features LRU-free efficient vectorization, automatic category detection, and comprehensive logging with intensity distribution reporting.

## Dependencies
pandas, numpy, tqdm, logging, dataclasses, typing

In [ ]:
from typing import List, Dict, Set, Optional, Tuple
from dataclasses import dataclass, field
from collections import OrderedDict
import logging

import pandas as pd
import numpy as np
from tqdm import tqdm

# Logging configuration
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
# CONFIGURATION

@dataclass
class SurpriseConfig:
    """Configuration for surprise labeling."""
    input_file: str = 'weighted_comments.csv'
    output_file: str = 'surprise_dataset.csv'

    # Thresholds for intensity levels
    surprise_high_threshold: float = 3.0
    total_high_threshold: float = 6.0
    total_medium_threshold: float = 3.0
    surprise_medium_threshold: float = 2.0
    total_low_threshold: float = 1.75

    # Category restrictions
    restricted_categories: Set[str] = field(default_factory=lambda: {
        'Странность/Отклонения', 'Интерес'
    })

    # Support map for surprise detection
    support_map: Dict[str, List[str]] = field(default_factory=lambda: {
        'Странность/Отклонения': ['Интерес', 'Удивление', 'Неожиданность', 'Отрицание'],
        'Неожиданность': ['Странность/Отклонения', 'Удивление', 'Интерес', 'Отрицание'],
        'Интерес': ['Странность/Отклонения', 'Удивление', 'Восхищение/Впечатление',
                    'Неожиданность', 'Отрицание'],
        'Чудо природы': ['Странность/Отклонения', 'Интерес', 'Отрицание', 'Удивление',
                         'Травмы', 'Неожиданность', 'Восхищение/Впечатление'],
        'Травмы': ['Странность/Отклонения', 'Интерес', 'Удивление', 'Восхищение/Впечатление',
                   'Неожиданность', 'Чудо природы', 'Отрицание'],
        'Отрицание': ['Странность/Отклонения', 'Интерес', 'Удивление', 'Неожиданность',
                      'Травмы', 'Чудо природы']
    })

In [ ]:
# DATA CLEANING

def clean_text_column(df: pd.DataFrame, column: str = 'text') -> pd.DataFrame:
    """Clean text column from all types of line breaks."""
    df[column] = df[column].fillna('').astype(str)
    df[column] = df[column].str.replace(r'\\n|\\r|\\t|\n|\r|\t', ' ', regex=True)
    df[column] = df[column].str.replace(r'\s+', ' ', regex=True)
    df[column] = df[column].str.strip()
    return df

In [ ]:
# SURPRISE DETECTION

def get_active_categories(row: pd.Series, all_categories: List[str]) -> List[str]:
    """Get list of active categories from row."""
    return [cat for cat in all_categories if row[f'weight_{cat}'] > 0]


def has_support_for_surprise(row: pd.Series, support_map: Dict[str, List[str]]) -> bool:
    """Check if there is support for surprise through other categories."""
    if row['weight_Удивление'] <= 0:
        return False

    for dominant, support_list in support_map.items():
        if dominant in ['Удивление', 'Изумление']:
            continue

        if row[f'weight_{dominant}'] > 0:
            if 'Удивление' in support_list or 'Изумление' in support_list:
                for cat in support_list:
                    if cat not in ['Удивление', 'Изумление'] and row[f'weight_{cat}'] > 0:
                        return True

    return False


def determine_surprise(
    row: pd.Series,
    all_categories: List[str],
    support_map: Dict[str, List[str]],
    total_low_threshold: float
) -> int:
    """
    Determine if comment contains surprise (binary labeling).
    Returns 1 if surprise present, 0 otherwise.
    """
    active_cats = get_active_categories(row, all_categories)

    if row['weight_Изумление'] > 0 or row['weight_Удивление'] > 0:
        return 1

    if set(active_cats) == {'Странность/Отклонения'}:
        return 0

    for dominant, support_list in support_map.items():
        if row[f'weight_{dominant}'] > 0:
            if any(row[f'weight_{cat}'] > 0 for cat in support_list):
                return 1

    if row['total_weight'] >= total_low_threshold:
        return 1

    return 0

In [ ]:
# INTENSITY LABELING

def check_intensity_restrictions(
    active_cats: List[str],
    restricted_categories: Set[str]
) -> Tuple[bool, bool]:
    """
    Check if active categories contain restricted categories.
    Returns (has_restricted, has_other).
    """
    has_restricted = any(cat in restricted_categories for cat in active_cats)
    has_other = any(cat not in restricted_categories for cat in active_cats)
    return has_restricted, has_other


def determine_intensity_level(
    row: pd.Series,
    active_cats: List[str],
    config: SurpriseConfig
) -> int:
    """
    Determine surprise intensity level (1-3).
    Returns 0 if no surprise.
    """
    has_restricted, has_other = check_intensity_restrictions(
        active_cats, config.restricted_categories
    )

    # Level 3: Strong surprise
    if row['weight_Изумление'] > 0:
        return 3

    if row['weight_Удивление'] >= config.surprise_high_threshold:
        if not (has_restricted and not has_other):
            return 3

    if row['total_weight'] >= config.total_high_threshold:
        if not (has_restricted and not has_other):
            return 3

    # Level 2: Medium surprise
    if has_restricted and has_other:
        if (has_support_for_surprise(row, config.support_map) or
            row['weight_Удивление'] >= config.surprise_medium_threshold or
            row['total_weight'] >= config.total_medium_threshold):
            return 2
        return 1

    if has_support_for_surprise(row, config.support_map):
        return 2

    if row['weight_Удивление'] >= config.surprise_medium_threshold:
        return 2

    if row['total_weight'] >= config.total_medium_threshold:
        return 2

    # Level 1: Weak surprise
    return 1


def determine_surprise_intensity(
    row: pd.Series,
    all_categories: List[str],
    config: SurpriseConfig
) -> int:
    """
    Determine surprise intensity (0-3) for a comment.
    """
    if determine_surprise(row, all_categories, config.support_map, config.total_low_threshold) == 0:
        return 0

    active_cats = get_active_categories(row, all_categories)

    # Only Странность/Отклонения and/or Интерес - max level 1
    if set(active_cats).issubset(config.restricted_categories):
        return 1

    return determine_intensity_level(row, active_cats, config)

In [ ]:
# MAIN PIPELINE

def load_and_clean_data(config: SurpriseConfig) -> pd.DataFrame:
    """Load and clean input data."""
    try:
        df = pd.read_csv(config.input_file)
        logger.info(f"Loaded {len(df)} records from {config.input_file}")

        # Clean with progress bar
        logger.info("Cleaning text...")
        tqdm.pandas(desc="Cleaning text", ncols=80, mininterval=0.5)
        df = clean_text_column(df, 'text')

        return df
    except FileNotFoundError:
        logger.error(f"Input file not found: {config.input_file}")
        raise


def extract_category_columns(df: pd.DataFrame) -> List[str]:
    """Extract category names from weight columns."""
    category_columns = [col for col in df.columns if col.startswith('weight_')]
    return [col.replace('weight_', '') for col in category_columns]


def apply_binary_labeling(df: pd.DataFrame, all_categories: List[str], config: SurpriseConfig) -> pd.DataFrame:
    """Apply binary surprise labeling with progress bar."""
    tqdm.pandas(
        desc="Binary labeling",
        ncols=80,
        bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]',
        mininterval=0.5,
        miniters=len(df)//100 if len(df) > 100 else 1
    )
    df['is_surprise'] = df.progress_apply(
        lambda row: determine_surprise(
            row, all_categories, config.support_map, config.total_low_threshold
        ),
        axis=1
    )

    surprise_count = df['is_surprise'].sum()
    logger.info(f"Binary labeling complete: {surprise_count} comments with surprise ({surprise_count/len(df)*100:.1f}%)")
    return df


def apply_intensity_labeling(df: pd.DataFrame, all_categories: List[str], config: SurpriseConfig) -> pd.DataFrame:
    """Apply surprise intensity labeling with progress bar."""
    tqdm.pandas(
        desc="Intensity labeling",
        ncols=80,
        bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]',
        mininterval=0.5,
        miniters=len(df)//100 if len(df) > 100 else 1
    )
    df['surprise_intensity'] = df.progress_apply(
        lambda row: determine_surprise_intensity(row, all_categories, config),
        axis=1
    )

    # Log distribution
    intensity_dist = df[df['surprise_intensity'] > 0]['surprise_intensity'].value_counts().sort_index()
    logger.info("Intensity distribution:")
    for level, count in intensity_dist.items():
        logger.info(f"  Level {level}: {count} ({count/len(df)*100:.1f}%)")

    return df


def main() -> None:
    """Main execution function."""
    config = SurpriseConfig()

    df = load_and_clean_data(config)

    all_categories = extract_category_columns(df)
    logger.info(f"Found {len(all_categories)} categories: {', '.join(all_categories)}")

    logger.info("\nApplying binary surprise labeling...")
    df = apply_binary_labeling(df, all_categories, config)

    logger.info("\nApplying surprise intensity labeling...")
    df = apply_intensity_labeling(df, all_categories, config)

    # Final cleanup
    logger.info("Final text cleanup...")
    df = clean_text_column(df, 'text')

    # Save results
    logger.info(f"Saving results to {config.output_file}...")
    df.to_csv(config.output_file, index=False, quoting=0)

if __name__ == "__main__":
    main()